In [ ]:
!pip install "ray[tune]"

In [ ]:
import numpy as np
import kagglehub
import os
from glob import glob
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import ray
from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

In [ ]:
path = kagglehub.dataset_download("ayush1220/cifar10")
path_test = path + "/cifar10/test"
path_train = path + "/cifar10/train"

Using Colab cache for faster access to the 'cifar10' dataset.


In [ ]:
def load_images_from_classes(base_path, class_folders, class_to_idx,
                             flatten=True, normalize=True):
    """
    Wczytuje obrazki ze wszystkich klas w danym folderze
    """
    all_images = []
    all_labels = []

    for class_name in class_folders:
        class_path = os.path.join(base_path, class_name)
        class_idx = class_to_idx[class_name]

        # Znajdź wszystkie PNG w tym folderze
        image_files = glob(os.path.join(class_path, "*.png"))

        print(f"  📂 {class_name}: {len(image_files)} obrazków")

        # Wczytaj każdy obrazek
        for img_path in image_files:
            # Wczytaj obrazek
            img = Image.open(img_path)

            # Konwertuj do RGB (na wypadek RGBA lub grayscale)
            if img.mode != 'RGB':
                img = img.convert('RGB')

            # Konwertuj do numpy array
            img_array = np.array(img)

            # Normalizacja
            if normalize:
                img_array = img_array.astype('float32') / 255.0

            # Spłaszczenie jeśli potrzebne
            if flatten:
                # Z (32, 32, 3) -> (3072,)
                img_array = img_array.flatten()

            all_images.append(img_array)
            all_labels.append(class_idx)

    # Konwertuj do numpy arrays
    X = np.array(all_images)
    y = np.array(all_labels)

    print(f"  ✓ Wczytano: {X.shape}")

    return X, y

In [ ]:
def load_and_split_data(base_path, val_split=0.2):
    """
    Wczytuje dane i dzieli train na train+validation

    Args:
        base_path: ścieżka do folderu z train/test
        val_split: procent danych treningowych przeznaczonych na walidację
    """
    train_path = os.path.join(base_path, 'train')
    test_path = os.path.join(base_path, 'test')

    # Pobierz nazwy klas
    class_folders = sorted([d for d in os.listdir(train_path)
                           if os.path.isdir(os.path.join(train_path, d))])

    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_folders)}

    print(f"🏷️  Znalezione klasy ({len(class_folders)}): {class_folders}\n")

    # Wczytaj train
    print("📥 Wczytywanie danych treningowych...")
    X_train_full, y_train_full = load_images_from_classes(
        train_path, class_folders, class_to_idx, flatten=True, normalize=True
    )

    # Wczytaj test
    print("\n📥 Wczytywanie danych testowych...")
    X_test, y_test = load_images_from_classes(
        test_path, class_folders, class_to_idx, flatten=True, normalize=True
    )

    # Podział train na train + validation
    print(f"\n✂️  Dzielenie na train/validation (val={val_split*100}%)...")

    n_total = len(X_train_full)
    n_val = int(n_total * val_split)
    n_train = n_total - n_val

    # Losowe permutacje dla mieszania danych
    indices = np.random.permutation(n_total)
    train_indices = indices[:n_train]
    val_indices = indices[n_train:]

    X_train = X_train_full[train_indices]
    y_train = y_train_full[train_indices]
    X_val = X_train_full[val_indices]
    y_val = y_train_full[val_indices]

    print(f"  ✓ Train:      {X_train.shape}")
    print(f"  ✓ Validation: {X_val.shape}")
    print(f"  ✓ Test:       {X_test.shape}")

    return X_train, y_train, X_val, y_val, X_test, y_test, class_folders

In [ ]:
# X_train, y_train, X_val, y_val, X_test, y_test, class_names = load_and_split_data(path+'/cifar10')

#Definition of the network

In [ ]:
class HighwayLayer(nn.Module):
    def __init__(self, size, activation=F.relu):
        super(HighwayLayer, self).__init__()
        self.activation = activation
        self.normal_layer = nn.Linear(size, size)
        self.gate_layer = nn.Linear(size, size)

    def forward(self, x):
        normal = self.activation(self.normal_layer(x))
        gate = torch.sigmoid(self.gate_layer(x))
        output = normal * gate + x * (1 - gate)
        return output


class HighwayNetwork(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.5, batch_norm=True, ):
        super(HighwayNetwork, self).__init__()

        self.input_layer = nn.Linear(input_size, hidden_size)
        self.dropout = nn.Dropout(dropout)

        self.highway_layers = nn.ModuleList([
            HighwayLayer(hidden_size) for _ in range(num_layers)
        ])

        self.output_layer = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = F.relu(self.input_layer(x))
        x = self.dropout(x)

        for highway_layer in self.highway_layers:
            x = highway_layer(x)

        output = self.output_layer(x)
        return output

Image dataset definition

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Hiperparametryzacja

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Trenowanie jednej epoki"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc="Training", leave=False, ncols=80):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total

    return epoch_loss, epoch_acc


In [ ]:
def validate(model, val_loader, criterion, device):
    """Walidacja modelu"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc="Validation", leave=False, ncols=80):
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / len(val_loader)
    val_acc = 100. * correct / total

    return val_loss, val_acc

In [ ]:
# Funkcja trainable dla Ray Tune
def hiperparameter_highway(config):
    # Inicjalizacja urządzenia
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Przyklad – tu wczytaj swoje dane, np. MNIST, CIFAR itp.
    transform = transforms.ToTensor()
    # train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
    # val_dataset = datasets.MNIST(root="data", train=False, download=True, transform=transform)

    # train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    # val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=False)

    X_train, y_train, X_val, y_val, X_test, y_test, class_names = load_and_split_data(path+'/cifar10')

    train_dataset = ImageDataset(X_train, y_train)
    val_dataset = ImageDataset(X_val, y_val)
    test_dataset = ImageDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config["batch_size"], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=True)


    # Inicjalizacja modelu na podstawie config
    model = HighwayNetwork(
        input_size=X_train.shape[1],  # Zastąp swoją wartością, np. 784 dla MNIST
        hidden_size=config["hidden_size"],
        num_layers=config["num_layers"],
        output_size=10,  # Zastąp, np. 10 dla klasyfikacji 10 klas
        dropout=config["dropout"]
    ).to(device)

    # Kryterium i optymalizator
    criterion = nn.CrossEntropyLoss()  # Dostosuj jeśli potrzeba
    if config["optimizer"] == "adam":
        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    elif config["optimizer"] == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=config["lr"], momentum=0.9)
    else:
        raise ValueError("Nieznany optimizer")

    # Pętla treningowa
    for epoch in range(config["num_epochs"]):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)

        # Walidacja (opcjonalna)
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        # Raportuj metryki do Ray Tune (użyj val_acc jako głównej metryki do optymalizacji)
        tune.report({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

In [ ]:
search_space = {
    "lr": tune.loguniform(1e-4, 1e-2),
    "hidden_size": tune.choice([128, 256, 512]),
    "num_layers": tune.randint(1, 5),
    "dropout": tune.uniform(0.1, 0.5),
    "optimizer": tune.choice(["adam", "sgd"]),
    "batch_size": tune.choice([32, 64, 128]),
    "num_epochs": 50
}

scheduler = ASHAScheduler(
  metric="val_acc",
  mode="max",
  max_t=10,
  grace_period=1,
  reduction_factor=2
)

result = tune.run(
    hiperparameter_highway,
    config=search_space,
    num_samples=10,
    scheduler=scheduler
)


+-------------------------------------------------------------------------------+
| Configuration for experiment     hiperparameter_highway_2025-11-18_14-19-16   |
+-------------------------------------------------------------------------------+
| Search algorithm                 BasicVariantGenerator                        |
| Scheduler                        AsyncHyperBandScheduler                      |
| Number of trials                 10                                           |
+-------------------------------------------------------------------------------+

View detailed results here: /root/ray_results/hiperparameter_highway_2025-11-18_14-19-16
To visualize your results with TensorBoard, run: `tensorboard --logdir /tmp/ray/session_2025-11-18_13-12-09_712193_978/artifacts/2025-11-18_14-19-16/hiperparameter_highway_2025-11-18_14-19-16/driver_artifacts`

Trial status: 10 PENDING
Current time: 2025-11-18 14:19:16. Total running time: 0s
Logical resource usage: 0/2 CPUs, 0/0 GPUs

2025-11-18 14:20:01,109	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2025-11-18 14:20:01,118	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/hiperparameter_highway_2025-11-18_14-19-16' in 0.0077s.


Trial status: 2 RUNNING | 8 PENDING
Current time: 2025-11-18 14:20:01. Total running time: 44s
Logical resource usage: 2.0/2 CPUs, 0/0 GPUs
+-------------------------------------------------------------------------------------------------------------------------------------+
| Trial name                           status              lr     hidden_size     num_layers     dropout   optimizer       batch_size |
+-------------------------------------------------------------------------------------------------------------------------------------+
| hiperparameter_highway_90c32_00000   RUNNING    0.000157881             128              4    0.492684   sgd                    128 |
| hiperparameter_highway_90c32_00001   RUNNING    0.00262281              128              1    0.121707   sgd                     32 |
| hiperparameter_highway_90c32_00002   PENDING    0.00158589              256              2    0.343016   sgd                     32 |
| hiperparameter_highway_90c32_00003   PENDI

(hiperparameter_highway pid=17870) [2025-11-18 14:20:01,619 E 17870 17967] core_worker_process.cc:825: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


(hiperparameter_highway pid=17870)   📂 bird: 5000 obrazków
(hiperparameter_highway pid=17914)   📂 automobile: 5000 obrazków
(hiperparameter_highway pid=17870)   📂 cat: 5000 obrazków
(hiperparameter_highway pid=17914)   📂 bird: 5000 obrazków


2025-11-18 14:20:11,164	WARNING tune.py:1056 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: tune.run(..., resume=True)
2025-11-18 14:20:11,187	WARNING experiment_analysis.py:180 -- Failed to fetch metrics for 8 trial(s):
- hiperparameter_highway_90c32_00002: FileNotFoundError('Could not fetch metrics for hiperparameter_highway_90c32_00002: both result.json and progress.csv were not found at /root/ray_results/hiperparameter_highway_2025-11-18_14-19-16/hiperparameter_highway_90c32_00002_2_batch_size=32,dropout=0.3430,hidden_size=256,lr=0.0016,num_layers=2,optimizer=sgd_2025-11-18_14-19-16')
- hiperparameter_highway_90c32_00003: FileNotFoundError('Could not fetch metrics for hiperparameter_highway_90c32_00003: both result.json and progress.csv were not found at /root/ray_results/hiperparameter_highway_2025-11-18_14-19-16/hiperparameter_highway_90c32_00003_3_batch_size=64,dropout=0.3289,hidden_size=128,lr=0.0044,num_layers=1,optimizer=adam_